In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [2]:
train = pd.read_csv('./train.csv.xz').fillna(0)
test = pd.read_csv('./test.csv.xz').fillna(0)

In [3]:
train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [4]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,0,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,0,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,0,S


In [5]:
def sex2num(x):
    if x['Sex'] == 'male': return 1
    return 0

def embark2num(x):
    options = {'S':1, 'C':2, 'Q':3, 0:0}
    val = x['Embarked']
    return options[val]


In [6]:
train['Age'] = pd.cut(x=train['Age'],bins=4,labels=False)
train['Fare'] = pd.cut(x=train['Fare'],bins=4,labels=False)
train['Sex'] = train.apply(func=sex2num,axis=1)
train['Embarked'] = train.apply(func=embark2num,axis=1)

test['Age'] = pd.cut(x=test['Age'],bins=4,labels=False)
test['Fare'] = pd.cut(x=test['Fare'],bins=4,labels=False)
test['Sex'] = test.apply(func=sex2num,axis=1)
test['Embarked'] = test.apply(func=embark2num,axis=1)

In [7]:
train.loc[:,['Survived','Pclass','Sex','Age','SibSp','Fare','Embarked']].head()

,Survived,Pclass,Sex,Age,SibSp,Fare,Embarked
0,0,3,1,1,1,0,1
1,1,1,0,1,1,0,2
2,1,3,0,1,0,0,1
3,1,1,0,1,1,0,1
4,0,3,1,1,0,0,1


In [8]:
X_train, y_train = train.loc[:,['Pclass','Sex','Age','SibSp','Fare','Embarked']], train['Survived']
X_test = test.loc[:,['Pclass','Sex','Age','SibSp','Fare','Embarked']]

In [18]:
clf = GaussianNB()
y_pred = clf.fit(X_train, y_train).predict(X_train)
clf.score(X_train, y_train)
#accuracy_score(y_train, y_pred)

0.7912457912457912

In [10]:
clf = AdaBoostClassifier(n_estimators=10)
scores = cross_val_score(clf, X_train, y_train, cv=5)
scores.mean()

0.7935070982311785

In [23]:
clf = RandomForestClassifier(n_estimators=50, max_depth=None, min_samples_split=2, random_state=0)
scores = cross_val_score(clf, X_train, y_train, cv=5)
scores.mean()

0.8058603007957693

In [12]:
clf = LogisticRegression(solver='lbfgs')
y_pred = clf.fit(X_train, y_train).predict(X_train)
clf.score(X_train, y_train)

0.7968574635241302

In [13]:
clf = KNeighborsClassifier(n_neighbors=5, algorithm='kd_tree')
y_pred = clf.fit(X_train, y_train).predict(X_train)
clf.score(X_train, y_train)

0.8181818181818182

In [24]:
# export
clf = RandomForestClassifier(n_estimators=50, max_depth=None, min_samples_split=2, random_state=0)
test_y_pred = clf.fit(X_train, y_train).predict(X_test)

In [20]:
X_test.head()

,Pclass,Sex,Age,SibSp,Fare,Embarked
0,3,1,1,0,0,3
1,3,0,2,1,0,1
2,2,1,3,0,0,3
3,3,1,1,0,0,1
4,3,0,1,1,0,1


In [25]:
export_dt = pd.DataFrame({'PassengerId':test['PassengerId'], 'Survived':test_y_pred})

In [26]:
export_dt.to_csv('RandomForest_submission.csv',index=False)